# Run LLMs Locally on CPU — Complete Tutorial

This notebook walks through the full pipeline to run open-source LLMs on CPU:
1. Choose a model
2. Download it from Hugging Face
3. Convert + quantize with llama.cpp
4. Run inference and benchmark directly

## Pipeline Architecture

---

## Windows users: Set up WSL first

This tutorial runs best on **Ubuntu/Linux**. On Windows, use **WSL (Windows Subsystem for Linux)**.

### 1. Install WSL
Open **PowerShell as Administrator** and run:

```powershell
wsl --install
```

Then restart your machine. Ubuntu is usually installed by default.

### 2. Connect VS Code to WSL
- Install the **WSL** extension in VS Code
- Click the green button in the bottom-left corner
- Choose **Connect to WSL**

From this point on, all commands below should be run inside **Ubuntu**.

### 3. clone my repository
- Open a terminal in vscode
- git clone https://github.com/stevo32800/llm_to_cpu.git

### 4. Install system packages on Ubuntu

```bash
sudo apt update
sudo apt install -y python3 python3-pip python3-venv git build-essential g++ cmake
```

### 5. Create and activate a virtual environment

```bash
python3 -m venv .venv
source .venv/bin/activate
```

### 6. Upgrade pip and install the project dependencies

```bash
python -m pip install --upgrade pip
pip install -r requirements.txt
```

> If you reopen the terminal later, reactivate the environment with:
>
> ```bash
> source .venv/bin/activate
> ```

## Create a Hugging Face account
**1. Sign up:** https://huggingface.co/

**2. Create a token:** https://huggingface.co/settings/tokens

**3. Add your token to the `.env` file**

## 1. Choose Your Model


| Rank | Model | Size | Architecture | Main strengths | HuggingFace ID |
|------|-------|------|--------------|----------------|----------------|
| **1** | **Nanbeige4.1‑3B** | 3B | Dense | Very strong reasoning, agentic, massive tool‑use, sometimes surpasses 30B models | `Nanbeige/Nanbeige4.1-3B` |
| **2** | **Qwen3‑4B‑Thinking** | 4B | Dense + Thinking | Explicit CoT, 262K context, top‑tier reasoning | `Qwen/Qwen3-4B-Thinking-2507` |
| **3** | **Qwen2.5‑7B‑Instruct** | 7B | Dense | Best 7B generalist, excellent tool‑use, very stable | `Qwen/Qwen2.5-7B-Instruct` |
| **4** | **Phi‑4‑mini‑instruct** | 3.8B | Dense | Strong reasoning, compact, near 7B quality on many tasks | `microsoft/Phi-4-mini-instruct` |
| **5** | **Gemma‑4 E4B** | ~4B active | MoE | High quality, 128K context, efficient MoE | `google/gemma-4-E4B-it` |
| **6** | **Ministral‑3B‑Instruct** | 3B | Dense | Very efficient, edge‑optimized, best perf/efficiency ratio | `mistralai/Ministral-3-3B-Instruct-2512` |
| **7** | **rnj‑1‑instruct** | 8B | Dense | Agentic, STEM, strong tool‑calling, limited public benchmarks | `EssentialAI/rnj-1-instruct` |
| **8** | **Qwen2.5‑Coder‑1.5B‑Instruct** | 1.5B | Dense | Ultra‑fast, excellent small code model | `Qwen/Qwen2.5-Coder-1.5B-Instruct` |


## 2. Configuration

In [9]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))
from cpu_optimizer import cpu_optimizer
from huggingface_hub import snapshot_download

# ── Paths ─────────────────────────────────────────────────────────────────────────
BASE_DIR  = Path("./models")
FULL_DIR  = BASE_DIR / "full"
GGUF_DIR  = BASE_DIR / "gguf"
LLAMA_DIR = BASE_DIR / "llama.cpp"

# ── Quantization profiles ──────────────────────────────────────────────────────
QUANT_PROFILES = {
    "precision": "Q6_K",    # best quality, more RAM
    "balanced":  "Q4_K_M",  # good tradeoff — default
    "speed":     "Q3_K_M",  # fastest, less RAM
}

# ── Edit these two lines ────────────────────────────────────────────────────────────────
MODEL_NAME    = "mistralai/Ministral-3-3B-Instruct-2512"  # HuggingFace model ID
QUANT_PROFILE = "balanced"                              # precision | balanced | speed

QUANT_TYPE = QUANT_PROFILES[QUANT_PROFILE]
for d in [FULL_DIR, GGUF_DIR]: d.mkdir(parents=True, exist_ok=True)

print(f"Model   : {MODEL_NAME}")
print(f"Profile : {QUANT_PROFILE}  ->  {QUANT_TYPE}")


Model   : mistralai/Ministral-3-3B-Instruct-2512
Profile : balanced  ->  Q4_K_M


## 3. System Info

In [10]:
from utils import get_system_info

info = get_system_info()
print(f"CPU    : {info['cpu_brand']}")
print(f"Cores  : {info['physical_cores']}P / {info['logical_cores']}L")
print(f"RAM    : {info['ram_total_gb']} GB total  |  {info['ram_avail_gb']} GB available")
print(f"SIMD   : {info['simd_flags'] or ['none detected']}")


CPU    : AMD Ryzen 7 260 w/ Radeon 780M Graphics
Cores  : 8P / 16L
RAM    : 23.5 GB total  |  20.3 GB available
SIMD   : ['avx2', 'avx512f', 'avx512vnni']


In [11]:
# SIMD build check

from utils import check_llama_simd, print_simd_fix

cpu_has, build_has, missing = check_llama_simd()

print(f"CPU   : AVX2={cpu_has['avx2']}  AVX-512={cpu_has['avx512f']}  AMX={cpu_has['amx_tile']}")
print(f"Build : AVX2={build_has['avx2']}  AVX-512={build_has['avx512f']}  AMX={build_has['amx_tile']}")

if missing:
    print_simd_fix(missing)
else:
    print("\n✅ llama-cpp-python compiled with the correct SIMD flags.")
    print("   Source: https://github.com/abetlen/llama-cpp-python")


CPU   : AVX2=True  AVX-512=True  AMX=False
Build : AVX2=True  AVX-512=True  AMX=False

✅ llama-cpp-python compiled with the correct SIMD flags.
   Source: https://github.com/abetlen/llama-cpp-python


## 4. Download the Model

In [12]:
from dotenv import load_dotenv
import os
load_dotenv()

token = os.getenv("HF_TOKEN")

dest = FULL_DIR / MODEL_NAME

if dest.exists() and any(dest.iterdir()):
    print(f"Already downloaded: {dest}")
else:
    print(f"Downloading {MODEL_NAME} ...")
    snapshot_download(
        repo_id=MODEL_NAME,
        local_dir=str(dest),
        token=token,
        ignore_patterns=["*.msgpack", "*.h5", "flax_model*", "tf_model*"],
    )
    print(f"Saved to: {dest}")

Already downloaded: models/full/mistralai/Ministral-3-3B-Instruct-2512


## 5. Quantize with `cpu_optimizer`

The `cpu_optimizer` class (from `cpu_optimizer.py`) automates the full conversion pipeline:
- Clones & compiles `llama.cpp` with the right SIMD flags for your CPU
- Converts the HuggingFace model to GGUF F16
- Quantizes to the target format
- Cleans up the intermediate F16 file

### GGUF Formats: Speed, RAM, and Actual Quality

> **Note:** K-quant formats (Q4_K_M, Q5_K_M, Q6_K) are not exactly the advertised bit-width.
> They use hierarchical super-blocks where critical layers (attention, FFW) are promoted to Q6_K.
> Q4_K_M is actually ~4.9 bits/weight on an 8B model, not 4.0.
>
> Sources: [llama.cpp quantize README](https://github.com/ggml-org/llama.cpp/blob/master/tools/quantize/README.md) · [arXiv 2601.14277 — perplexity benchmarks](https://arxiv.org/abs/2601.14277)

| Profile | Type | Actual bits (8B) | RAM (3B) | RAM (7B) | Perplexity (WikiText-2) | Use case |
|---------|------|-----------------|----------|----------|------------------------|----------|
| precision | Q6_K | 6.56 | ~2.9 GB | ~6.1 GB | ~7.35 | Production, max quality |
| **balanced** | **Q4_K_M** | **4.89** | **~2.0 GB** | **~4.6 GB** | **7.56** | **Recommended — best trade-off** |
| speed | Q3_K_M | ~3.5 | ~1.6 GB | ~3.7 GB | ~8.0 | Very tight RAM |
| — | Q8_0 | 8.50 | ~3.3 GB | ~7.9 GB | 7.33 | Near-identical to FP16 |
| — | F16 | 16.00 | ~6.5 GB | ~14.9 GB | 7.32 (baseline) | Reference only, not practical on CPU |

**Takeaway:** Q4_K_M loses only 0.24 perplexity points vs FP16 — imperceptible in practice. Q3_K_M starts to degrade response coherence.


In [13]:
import subprocess

optimizer = cpu_optimizer(
    model_name=MODEL_NAME,
    input_path=str(FULL_DIR) + "/",
    output_path=str(GGUF_DIR) + "/",
    quant_type=QUANT_TYPE,
    quantization_verbose=False,
)
optimizer.llama_dir = LLAMA_DIR

model_folder = MODEL_NAME.split("/", 1)[1] if "/" in MODEL_NAME else MODEL_NAME
f16_path   = GGUF_DIR / f"{model_folder}-f16.gguf"
quant_path = GGUF_DIR / f"{model_folder}-{QUANT_TYPE}.gguf"

# ── Case 1: already quantized ──────────────────────────────────────────────
if quant_path.exists():
    print(f"Already quantized: {quant_path.name} ({quant_path.stat().st_size/1024**3:.2f} GB)")
    gguf_path  = quant_path
    quant_used = QUANT_TYPE
    success    = True

# ── Case 2: F16 exists, skip conversion ───────────────────────────────────
elif f16_path.exists():
    print(f"F16 found ({f16_path.stat().st_size/1024**3:.1f} GB) — running quantization only")
    optimizer.gguf_f16 = f16_path
    optimizer._auto_detect_model_params()
    optimizer._select_quant_type()
    optimizer._setup_directories()

    cpu_info = optimizer._detect_cpu_capabilities()
    build_flags, build_threads = cpu_info[0], cpu_info[1]
    quant_threads = cpu_info[2] if len(cpu_info) > 2 else build_threads

    optimizer._clone_llama_cpp()
    optimizer._build_llama_cpp(build_flags, build_threads)

    quantize_bin = optimizer._get_quantize_bin()

    cmd = f'"{quantize_bin}" "{f16_path}" "{optimizer.gguf_quantized}" {QUANT_TYPE} {quant_threads}'
    subprocess.run(cmd, shell=True)  # don't use check=True — llama-quantize exits 1 on Windows even on success

    if optimizer.gguf_quantized.exists():
        optimizer._cleanup()
        gguf_path  = optimizer.gguf_quantized
        quant_used = QUANT_TYPE
        success    = True
    else:
        print("Quantization failed — output file not found")
        success = False

# ── Case 3: full pipeline ─────────────────────────────────────────────────
else:
    success, quant_used = optimizer.main_optimize()
    if success:
        gguf_path = optimizer.gguf_quantized

if success:
    print(f"\nGGUF : {gguf_path}")
    print(f"Size : {gguf_path.stat().st_size / 1024**3:.2f} GB")


🚀 Optimising model for CPU: mistralai/Ministral-3-3B-Instruct-2512

🔍 Auto-detecting model parameters from config.json...
   Size extracted from model name: 3.0B
✅ Detected parameters:
   Parameters : ~3.0B
   Layers     : 40
   Heads      : 8
   Head dim   : 640
   Max context: 131,072 tokens (131k)

🎯 Quantisation: Q4_K_M
📂 Source model : Ministral-3-3B-Instruct-2512
📂 GGUF output  : gguf
📂 llama.cpp    : llama.cpp

💾 Total RAM    : 23.5 GB
💾 Available RAM: 20.3 GB

📊 Model 3.0B Q4_K_M
  Estimated size    : 1.9 GB
  RAM after model   : ~18.4 GB
  KV cache estimate : 16.80 GB
  Context           : 22016 tokens (22k)

🖥️  CPU: AMD Ryzen 7 260 w/ Radeon 780M Graphics
  AVX2        : ✓
  AVX-512     : ✓
  AVX-512VNNI : ✓
  AMX         : ✗

  Physical cores : 8
  Logical cores  : 16

  Build flags   : -DGGML_NATIVE=ON -DGGML_OPENMP=ON -DGGML_F16C=ON -DGGML_AVX512=ON -DGGML_AVX512_VNNI=ON
  LTO           : ON
  Build threads : 6
  Quant threads : 16

🔄 Updating llama.cpp...

>>> git fetch

## 6. Load the Model and Run Inference

We use `llama-cpp-python` directly — no extra wrapper needed.

In [14]:
from utils import load_model, find_gguf

gguf_path = find_gguf(MODEL_NAME, GGUF_DIR)
n_ctx_hint = getattr(optimizer, 'ctx', None) if 'optimizer' in dir() else None

model, load_info = load_model(gguf_path, n_ctx=n_ctx_hint)


  GGUF detected: Ministral-3-3B-Instruct-2512-Q3_K_M.gguf
  Threads: 8  |  ctx: 22016 tokens


llama_context: setting new yarn_attn_factor = 1.0000 (mscale == 1.0, mscale_all_dim = 1.0)
llama_context: n_ctx_seq (22016) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


  Flash Attention + KV Q4_0 enabled

  Loaded in 2.6s  |  +0.7 GB RAM


## 7. Generate Text + Measure Performance

In [15]:
from utils import generate as _generate, warmup

SYSTEM_PROMPT = (
    "You are a helpful, concise assistant. "
    "Answer clearly and directly. "
    "If the question is about code, provide working examples."
)

def generate(prompt: str, max_tokens: int = 150, stream: bool = True) -> dict:
    return _generate(model=model, prompt=prompt,
                     system_prompt=SYSTEM_PROMPT,
                     max_tokens=max_tokens, stream=stream)

warmup(model)


Warming up... ready.


In [16]:
result = generate(
    prompt="Explain what a Large Language Model is in 3 sentences.",
    max_tokens=150,
    stream=True,
)
print(f"{result['tok_s']} tok/s  |  {result['tokens']} tokens  |  "
      f"gen {result['generation_time_s']}s  |  CPU {result['avg_cpu_pct']}%")


A **Large Language Model (LLM)** is an advanced AI system trained on vast amounts of text data to understand and generate human-like language.

It uses sophisticated algorithms, like transformer architectures, to predict and generate coherent responses based on input patterns it has learned.

Common examples include tools like me, powered by models like GPT-3 or its successors.
14.0 tok/s  |  71 tokens  |  gen 5.06s  |  CPU 51.2%
